# Examen ICCD753 Recuperación de Información

## 1. Objetivo

Desarrollar un sistema de recuperación de información sobre un corpus asignado. El sistema deberá indexar los documentos, recuperar y ordenar resultados mediante TF-IDF con similitud coseno y BM25, y exponer la búsqueda como un servicio web publicado en internet.

Se evaluará el servicio enviando consultas desde un cliente (p.ej. Postman) y contrastando los resultados con los juicios de relevancia (*qrels*) del corpus.

## 2. Corpus asignado

Cada estudiante recibirá un conjunto de datos. Todos contienen documentos, consultas y qrels. Se utilizarán los identificadores originales de los documentos para que los resultados puedan evaluarse.

| Corpus | Documentos y consultas | Qrels |
| --- | --- | --- |
| NFCorpus | [BeIR/nfcorpus](https://huggingface.co/datasets/BeIR/nfcorpus) | [BeIR/nfcorpus-qrels](https://huggingface.co/datasets/BeIR/nfcorpus-qrels) |
| SciFact | [BeIR/scifact](https://huggingface.co/datasets/BeIR/scifact) | [BeIR/scifact-qrels](https://huggingface.co/datasets/BeIR/scifact-qrels) |

## 3. Requisitos del sistema

### 3.1. Preparación e indexación

El sistema deberá:

1. Cargar e indexar los documentos del corpus asignado.
2. Aplicar y documentar las decisiones de preprocesamiento: tokenización, normalización, eliminación de stopwords y stemming o lematización.
3. Construir un vocabulario y un índice invertido que permita recuperar documentos.
4. Conservar la relación entre cada documento indexado y su identificador original.

La unidad de evaluación será el documento original, porque los qrels relacionan consultas con identificadores de documentos. No se debe dividir el texto en fragmentos.

### 3.2. Modelos de recuperación

Para cualquier consulta de texto, el sistema deberá producir dos rankings independientes:

- TF-IDF con similitud coseno.
- BM25.

Cada ranking deberá ordenar los documentos por puntuación de relevancia y permitir solicitar los primeros `k` resultados.

### 3.3. Servicio web

El sistema deberá estar publicado en internet y ofrecer, como mínimo, los siguientes endpoints:

**Comprobación de disponibilidad**

```http
GET /health
```

**Búsqueda**

```http
POST /search
Content-Type: application/json
```

Ejemplo de solicitud:

```json
{
  "query": "texto de la consulta",
  "model": "bm25",
  "top_k": 10
}
```

El campo `model` deberá aceptar `bm25` y `tfidf`. Una respuesta tendrá esta estructura:

```json
{
  "model": "bm25",
  "results": [
    {
      "rank": 1,
      "document_id": "identificador_original",
      "score": 8.42,
      "title": "Título, si está disponible",
      "snippet": "Fragmento del documento recuperado"
    }
  ]
}
```

Los resultados deberán estar ordenados de mayor a menor puntuación, no contener documentos duplicados y respetar el valor solicitado de `top_k`. El servicio debe aceptar consultas que el estudiante no conozca de antemano.

## 4. Evaluación de la recuperación

Cada estudiante debe evaluar ambos modelos con consultas de la partición `test` y sus qrels. Presentará, como mínimo:

- Precision@10.
- Recall@10.
- NDCG@10.
- Una comparación de los resultados de TF-IDF y BM25.

En el Jupyter Notebook se deberá indicar qué consultas se utilizaron y permitir reproducir los cálculos. La evaluación del docente podrá utilizar otras consultas del mismo conjunto.

## 5. Prueba del docente

El docente accederá a la URL entregada y realizará solicitudes desde un cliente de web service para comprobar:

1. La disponibilidad del servicio.
2. El funcionamiento de TF-IDF y BM25.
3. El formato, orden y validez de los resultados.
4. La correspondencia de los `document_id` con el corpus y los qrels.
5. La respuesta ante consultas no incluidas en los ejemplos de entrega.

## 6. Entregables

Cada estudiante entregará:

1. La URL pública del servicio web, accesible durante el período de revisión.
2. El código en Jupyter Notebook con los resultados de la ejecución de cada bloque de código, incluyendo los resultados de las métricas de evaluación de los modelos de recuperación.

Se evaluará que ambos modelos funcionen correctamente y que cada estudiante interprete sus resultados.

---

# Desarrollo

**Corpus asignado:** NFCorpus (`BeIR/nfcorpus` y `BeIR/nfcorpus-qrels`).

El código está en módulos para que el notebook y el servicio web usen lo mismo:

```
app.py                 # servicio FastAPI (/health y /search)
src/data.py            # carga de documentos, consultas y qrels
src/preprocessing.py   # tokenización, stopwords y stemming (NLTK)
src/index.py           # índice invertido propio
src/retrievers.py      # TF-IDF con coseno y BM25 propios
src/evaluation.py      # P@10, R@10 y NDCG@10
tests/test_api.py      # pruebas de la API
```

No uso `rank_bm25` ni librerías de búsqueda: todo está programado a mano.

In [1]:
import warnings; warnings.filterwarnings("ignore")

import json
import sys
import textwrap
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats

# Así el notebook encuentra `src` y `app` aunque se abra desde otra carpeta
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.data import (CORPUS_REVISION, QRELS_REVISION, load_corpus,
                      load_qrels, load_queries)
from src.evaluation import evaluate, summarize
from src.index import InvertedIndex
from src.preprocessing import preprocess, setup_nltk
from src.retrievers import BM25Retriever, TfidfRetriever

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)
setup_nltk()
print("Revisión BeIR/nfcorpus      :", CORPUS_REVISION)
print("Revisión BeIR/nfcorpus-qrels:", QRELS_REVISION)

Revisión BeIR/nfcorpus      : b5026a0e96e8a7ac4f95f482a596389289d46269
Revisión BeIR/nfcorpus-qrels: a451b3b26d3ae1358f259c1a3a4dd61fcea35a65


## 1. Carga y exploración de NFCorpus

Los datos se descargan de Hugging Face con `huggingface_hub` y se leen con `pyarrow` (corpus y consultas) y `csv` (los qrels son `.tsv`). Fijé la revisión de cada repositorio para que los datos sean siempre los mismos.

Los IDs son texto (`MED-10`, `PLAIN-2`), por eso nunca se convierten a número.

In [2]:
corpus = load_corpus()
queries = load_queries()
qrels = {split: load_qrels(split) for split in ("train", "dev", "test")}

print(f"Documentos en el corpus : {len(corpus)}")
print(f"Consultas (todas)       : {len(queries)}")
for split, q in qrels.items():
    print(f"Consultas con qrels en {split:<5}: {len(q)}")

ejemplo_id = next(iter(corpus))
print("\nEjemplo de documento:", repr(ejemplo_id), "->", type(ejemplo_id).__name__)
print("Título:", corpus[ejemplo_id].title)
print("Texto :", corpus[ejemplo_id].text[:200], "...")

Documentos en el corpus : 3633
Consultas (todas)       : 3237
Consultas con qrels en train: 2590
Consultas con qrels en dev  : 324
Consultas con qrels en test : 323

Ejemplo de documento: 'MED-10' -> str
Título: Statin Use and Breast Cancer Survival: A Nationwide Cohort Study from Finland
Texto : Recent studies have suggested that statins, an established drug group in the prevention of cardiovascular mortality, could delay or prevent breast cancer recurrence but the effect on disease-specific  ...


In [3]:
# Distribución de grados de relevancia en cada partición
filas = []
for split, q in qrels.items():
    scores = pd.Series([s for judged in q.values() for s in judged.values()])
    conteo = scores.value_counts().to_dict()
    por_consulta = [len(j) for j in q.values()]
    filas.append({
        "partición": split,
        "consultas": len(q),
        "pares juzgados": len(scores),
        "score=1": conteo.get(1, 0),
        "score=2": conteo.get(2, 0),
        "relevantes/consulta (media)": round(np.mean(por_consulta), 1),
        "relevantes/consulta (mediana)": int(np.median(por_consulta)),
    })
display(pd.DataFrame(filas).set_index("partición"))

# Longitud de los documentos (palabras sin procesar)
palabras = pd.Series([len(f"{d.title} {d.text}".split()) for d in corpus.values()])
print("\nPalabras por documento:")
print(palabras.describe().round(1).to_string())

,consultas,pares juzgados,score=1,score=2,relevantes/consulta (media),relevantes/consulta (mediana)
partición,,,,,,
train,2590,110575,110575,0,42.7,18
dev,324,11385,10864,521,35.1,18
test,323,12334,11758,576,38.2,16



Palabras por documento:
count    3633.0
mean      233.8
std        83.4
min        17.0
25%       185.0
50%       237.0
75%       273.0
max      1481.0


**Lo que se ve**

- `train` solo tiene `score=1`. `dev` y `test` también tienen `score=2` (relevancia directa, ~5 %).
- Cada consulta tiene ~38 documentos relevantes (mediana 16). Con solo 10 resultados, `Recall@10` y `Precision@10` no pueden ser cercanos a 1.
- Los documentos son abstracts médicos de largo parecido (mediana 237 palabras). Las consultas son cortas.
- Para `P@10` y `R@10` es relevante todo `score >= 1`. Para `NDCG@10` se usa el valor `score` (1 o 2).

## 2. Preprocesamiento e indexación

Mismos pasos para documentos y consultas:

1. **Minúsculas y tokenización** con `word_tokenize` (NLTK).
2. **Limpieza de puntuación:** `beta-carotene` pasa a `beta`, `carotene`. Los números se dejan (`b12`, `omega 3`) porque en medicina importan.
3. **Stopwords en inglés** (lista de NLTK): palabras como *the* o *of* están en todos los documentos y no ayudan a distinguirlos.
4. **Stemming de Porter (opcional):** junta variantes como `statins` y `statin`.

Cada documento se indexa completo como `título + " " + texto`, con su `document_id` original.

In [4]:
frase = "Do Cholesterol Statin Drugs Cause Breast Cancer? Vitamin B12 and beta-carotene intake in 5,000 women."
print("Original   :", frase)
print("Sin stem   :", preprocess(frase, stem=False))
print("Con Porter :", preprocess(frase, stem=True))

Original   : Do Cholesterol Statin Drugs Cause Breast Cancer? Vitamin B12 and beta-carotene intake in 5,000 women.
Sin stem   : ['cholesterol', 'statin', 'drugs', 'cause', 'breast', 'cancer', 'vitamin', 'b12', 'beta', 'carotene', 'intake', '5', '000', 'women']
Con Porter : ['cholesterol', 'statin', 'drug', 'caus', 'breast', 'cancer', 'vitamin', 'b12', 'beta', 'caroten', 'intak', '5', '000', 'women']


In [5]:
indices = {}
for stem in (False, True):
    t0 = time.perf_counter()
    indices[stem] = InvertedIndex.build(corpus, stem=stem)
    segundos = time.perf_counter() - t0
    idx = indices[stem]
    print(f"stem={stem!s:<5} | N={idx.N} | vocabulario={len(idx.df):>6} términos | "
          f"largo medio={np.mean(list(idx.doc_lengths.values())):.1f} tokens | "
          f"construcción={segundos:.1f} s")

# Efecto del stemming sobre palabras de ejemplo
for palabra in ("statin", "statins", "cancer", "cancers", "diets", "dietary"):
    print(f"{palabra:<8} df sin stem = {indices[False].df.get(palabra, 0):>4}", end=" | ")
    print("raíz Porter:", preprocess(palabra, stem=True))

# El índice conserva el document_id original de cada documento
termino = "statin"
posting = indices[True].postings[termino]
print(f"\nTérmino {termino!r}: aparece en {indices[True].df[termino]} documentos. Primeros 5 (document_id: tf):")
print(dict(list(posting.items())[:5]))

stem=False | N=3633 | vocabulario= 26341 términos | largo medio=165.8 tokens | construcción=3.3 s


stem=True  | N=3633 | vocabulario= 19050 términos | largo medio=165.8 tokens | construcción=3.4 s
statin   df sin stem =   16 | raíz Porter: ['statin']
statins  df sin stem =   22 | raíz Porter: ['statin']
cancer   df sin stem =  741 | raíz Porter: ['cancer']
cancers  df sin stem =  176 | raíz Porter: ['cancer']
diets    df sin stem =  293 | raíz Porter: ['diet']
dietary  df sin stem = 1029 | raíz Porter: ['dietari']

Término 'statin': aparece en 27 documentos. Primeros 5 (document_id: tf):
{'MED-10': 12, 'MED-14': 6, 'MED-870': 1, 'MED-1193': 11, 'MED-1207': 2}


**Por qué ayuda el stemming**

- Baja el vocabulario de ~26.000 a ~19.000 términos.
- Une plurales y sufijos (`statin`/`statins`, `cancer`/`cancers`, `diet`/`diets`). Así una consulta encuentra documentos que dicen lo mismo con otra forma de la palabra.
- Puede juntar palabras que no son iguales (`dietary` pasa a `dietari`), pero con consultas cortas el balance es bueno. Las métricas lo confirman más abajo.
- El índice se arma en memoria en ~3 segundos, no hace falta guardarlo en disco.

## 3. Modelos de recuperación

Los dos usan el mismo índice invertido. El ranking se ordena por `(-score, doc_id)`, sin duplicados, y solo salen documentos que comparten algún término con la consulta.

In [6]:
retrievers = {
    ("tfidf", False): TfidfRetriever(indices[False], stem=False),
    ("tfidf", True): TfidfRetriever(indices[True], stem=True),
    ("bm25", False): BM25Retriever(indices[False], stem=False),
    ("bm25", True): BM25Retriever(indices[True], stem=True),
}

def mostrar_ranking(qid: str, k: int = 5, stem: bool = True) -> None:
    print(f"Consulta {qid}: {queries[qid]!r}  "
          f"({sum(s >= 1 for s in qrels['test'][qid].values())} relevantes en qrels)")
    for modelo in ("tfidf", "bm25"):
        filas = []
        for rank, r in enumerate(retrievers[(modelo, stem)].search(queries[qid], k=k), start=1):
            filas.append({
                "rank": rank,
                "document_id": r.doc_id,
                "score": round(r.score, 4),
                "relevancia (qrels)": qrels["test"][qid].get(r.doc_id, 0),
                "título": corpus[r.doc_id].title[:75],
            })
        print(f"\n  {modelo.upper()} (stem={stem})")
        display(pd.DataFrame(filas).set_index("rank"))

mostrar_ranking("PLAIN-2")

Consulta PLAIN-2: 'Do Cholesterol Statin Drugs Cause Breast Cancer?'  (24 relevantes en qrels)

  TFIDF (stem=True)


,document_id,score,relevancia (qrels),título
rank,,,,
1,MED-10,0.3722,2,Statin Use and Breast Cancer Survival: A Nationwide Cohort Study from Finla
2,MED-2429,0.3491,2,Statin use and risk of breast cancer: a meta-analysis of observational stud
3,MED-14,0.3308,2,Statin use after diagnosis of breast cancer and survival: a population-base
4,MED-2431,0.2580,2,Long-term statin use and risk of ductal and lobular breast cancer among wom
5,MED-4830,0.2420,1,Statin therapy induces ultrastructural damage in skeletal muscle in patient



  BM25 (stem=True)


,document_id,score,relevancia (qrels),título
rank,,,,
1,MED-14,21.5766,2,Statin use after diagnosis of breast cancer and survival: a population-base
2,MED-10,21.5145,2,Statin Use and Breast Cancer Survival: A Nationwide Cohort Study from Finla
3,MED-2429,18.4896,2,Statin use and risk of breast cancer: a meta-analysis of observational stud
4,MED-2431,17.4734,2,Long-term statin use and risk of ductal and lobular breast cancer among wom
5,MED-1193,17.1258,0,The effects of lowering LDL cholesterol with statin therapy in people at lo


In [7]:
mostrar_ranking("PLAIN-102")

Consulta PLAIN-102: 'Stopping Heart Disease in Childhood'  (24 relevantes en qrels)

  TFIDF (stem=True)


,document_id,score,relevancia (qrels),título
rank,,,,
1,MED-3954,0.1614,0,Does childhood meat eating contribute to sex differences in risk factors fo
2,MED-4247,0.1481,0,Can lifestyle changes reverse coronary heart disease? The Lifestyle Heart T
3,MED-4616,0.1481,0,Can lifestyle changes reverse coronary heart disease? The Lifestyle Heart T
4,MED-4783,0.1254,0,Childhood soy intake and breast cancer risk in Asian American women.
5,MED-1991,0.1252,0,Plant foods and plant-based diets: protective against childhood obesity?



  BM25 (stem=True)


,document_id,score,relevancia (qrels),título
rank,,,,
1,MED-3954,13.1342,0,Does childhood meat eating contribute to sex differences in risk factors fo
2,MED-4247,10.6655,0,Can lifestyle changes reverse coronary heart disease? The Lifestyle Heart T
3,MED-4616,10.6655,0,Can lifestyle changes reverse coronary heart disease? The Lifestyle Heart T
4,MED-1999,10.5055,0,Strategies for preventing type 2 diabetes: an update for clinicians
5,MED-3253,10.5008,2,Pathobiological determinants of atherosclerosis in youth risk scores are as


El score de TF-IDF va de 0 a 1 (es un coseno) y el de BM25 no tiene tope. Por eso los scores de los dos modelos **no se comparan entre sí**, solo el orden del ranking.

## 4. Evaluación comparativa en `test`

Se evalúan **todas** las consultas de `test` con al menos un documento relevante (`score >= 1`). Métricas con corte en 10:

- **Precision@10:** relevantes en el top-10 / 10.
- **Recall@10:** relevantes en el top-10 / todos los relevantes de la consulta.
- **NDCG@10:** usa el `score` (1 o 2) como ganancia, con descuento $1/\log_2(pos+1)$ y normaliza con el ranking ideal.

Se promedia sobre las consultas.

In [8]:
test_qrels = qrels["test"]
ids_evaluados = sorted(
    qid for qid, j in test_qrels.items() if qid in queries and any(s >= 1 for s in j.values())
)
print(f"Consultas de test con qrels      : {len(test_qrels)}")
print(f"Consultas evaluables (>=1 relev.): {len(ids_evaluados)}")
print("\nIDs de consulta utilizados (orden alfabético):")
print(textwrap.fill(", ".join(ids_evaluados), width=110))

Consultas de test con qrels      : 323
Consultas evaluables (>=1 relev.): 323

IDs de consulta utilizados (orden alfabético):
PLAIN-1008, PLAIN-1018, PLAIN-102, PLAIN-1028, PLAIN-1039, PLAIN-1050, PLAIN-1066, PLAIN-1088, PLAIN-1098,
PLAIN-1109, PLAIN-1119, PLAIN-112, PLAIN-1130, PLAIN-1141, PLAIN-1151, PLAIN-1161, PLAIN-1172, PLAIN-1183,
PLAIN-1193, PLAIN-12, PLAIN-1203, PLAIN-1214, PLAIN-1225, PLAIN-123, PLAIN-1236, PLAIN-1249, PLAIN-1262,
PLAIN-1275, PLAIN-1288, PLAIN-1299, PLAIN-1309, PLAIN-1320, PLAIN-133, PLAIN-1331, PLAIN-1342, PLAIN-1353,
PLAIN-1363, PLAIN-1374, PLAIN-1387, PLAIN-1398, PLAIN-1409, PLAIN-1419, PLAIN-1429, PLAIN-143, PLAIN-1441,
PLAIN-1453, PLAIN-1463, PLAIN-1473, PLAIN-1485, PLAIN-1496, PLAIN-1506, PLAIN-1516, PLAIN-1527, PLAIN-153,
PLAIN-1537, PLAIN-1547, PLAIN-1557, PLAIN-1568, PLAIN-1579, PLAIN-1590, PLAIN-1601, PLAIN-1611, PLAIN-1621,
PLAIN-1635, PLAIN-1645, PLAIN-165, PLAIN-1656, PLAIN-1667, PLAIN-1679, PLAIN-1690, PLAIN-1700, PLAIN-1710,
PLAIN-1721, PLAIN-1

In [9]:
resultados = {}
filas = []
for modelo, nombre in (("tfidf", "TF-IDF"), ("bm25", "BM25")):
    for stem in (False, True):
        por_consulta = evaluate(retrievers[(modelo, stem)], queries, test_qrels, k=10)
        assert sorted(por_consulta) == ids_evaluados
        resultados[(modelo, stem)] = por_consulta
        filas.append({"modelo": f"{nombre} {'con' if stem else 'sin'} stemming",
                      **summarize(por_consulta)})

tabla = pd.DataFrame(filas).set_index("modelo").round(4)
display(tabla)

,P@10,R@10,NDCG@10
modelo,,,
TF-IDF sin stemming,0.2167,0.1443,0.3033
TF-IDF con stemming,0.2310,0.1526,0.3193
BM25 sin stemming,0.2211,0.1517,0.3117
BM25 con stemming,0.2406,0.1549,0.3297


In [10]:
# Comparación consulta por consulta: BM25 vs TF-IDF (ambos con stemming)
nd_bm25 = np.array([resultados[("bm25", True)][q]["NDCG@10"] for q in ids_evaluados])
nd_tfidf = np.array([resultados[("tfidf", True)][q]["NDCG@10"] for q in ids_evaluados])
dif = nd_bm25 - nd_tfidf
print(f"BM25 gana en {(dif > 1e-12).sum()} consultas | empatan en {(abs(dif) <= 1e-12).sum()} | "
      f"TF-IDF gana en {(dif < -1e-12).sum()}  (NDCG@10)")
print(f"Diferencia media de NDCG@10 = {dif.mean():+.4f}")
print("t pareada  : p =", round(stats.ttest_rel(nd_bm25, nd_tfidf).pvalue, 4))
print("Wilcoxon   : p =", round(stats.wilcoxon(nd_bm25, nd_tfidf).pvalue, 4))

# Efecto del stemming
for modelo in ("tfidf", "bm25"):
    s0 = summarize(resultados[(modelo, False)])["NDCG@10"]
    s1 = summarize(resultados[(modelo, True)])["NDCG@10"]
    print(f"{modelo.upper():<6} NDCG@10 sin stem {s0:.4f} -> con stem {s1:.4f} ({s1 - s0:+.4f})")

BM25 gana en 97 consultas | empatan en 174 | TF-IDF gana en 52  (NDCG@10)
Diferencia media de NDCG@10 = +0.0104
t pareada  : p = 0.0088
Wilcoxon   : p = 0.0008
TFIDF  NDCG@10 sin stem 0.3033 -> con stem 0.3193 (+0.0160)
BM25   NDCG@10 sin stem 0.3117 -> con stem 0.3297 (+0.0180)


In [11]:
# ¿Qué pasa con la longitud de los documentos que devuelve cada modelo?
largos = indices[True].doc_lengths
rel_largos = [largos[d] for q in ids_evaluados for d, s in test_qrels[q].items() if s >= 1 and d in largos]
print(f"Largo medio de todo el corpus        : {np.mean(list(largos.values())):6.1f} tokens")
print(f"Largo medio de los docs relevantes   : {np.mean(rel_largos):6.1f} tokens")
for modelo in ("tfidf", "bm25"):
    top = [largos[r.doc_id] for q in ids_evaluados
           for r in retrievers[(modelo, True)].search(queries[q], k=10)]
    print(f"Largo medio del top-10 de {modelo.upper():<6}     : {np.mean(top):6.1f} tokens")

# Parecido entre los dos rankings y consultas donde ambos fallan
solape = [len({r.doc_id for r in retrievers[("tfidf", True)].search(queries[q], 10)} &
              {r.doc_id for r in retrievers[("bm25", True)].search(queries[q], 10)}) / 10
          for q in ids_evaluados]
ambos_cero = [q for q in ids_evaluados if nd_bm25[ids_evaluados.index(q)] == 0
              and nd_tfidf[ids_evaluados.index(q)] == 0]
print(f"\nDocumentos en común en el top-10 (promedio): {np.mean(solape):.0%}")
print(f"Consultas donde ambos modelos tienen NDCG@10 = 0: {len(ambos_cero)} de {len(ids_evaluados)}")
print("Ejemplos:", ambos_cero[:4])

Largo medio de todo el corpus        :  165.8 tokens
Largo medio de los docs relevantes   :  166.7 tokens
Largo medio del top-10 de TFIDF      :  155.2 tokens


Largo medio del top-10 de BM25       :  163.0 tokens

Documentos en común en el top-10 (promedio): 64%
Consultas donde ambos modelos tienen NDCG@10 = 0: 95 de 323
Ejemplos: ['PLAIN-1008', 'PLAIN-1028', 'PLAIN-1050', 'PLAIN-1066']


In [12]:
# Un caso donde los dos fallan, para ver por qué
qid = ambos_cero[0]
print(f"Consulta {qid}: {queries[qid]!r}")
print("Tokens de la consulta:", preprocess(queries[qid], stem=True))
print("\nAlgunos documentos relevantes en qrels:")
for d, s in list(test_qrels[qid].items())[:3]:
    print(f"  [{s}] {d}: {corpus[d].title[:90]}")
print("\nTop-3 de BM25:")
for r in retrievers[("bm25", True)].search(queries[qid], k=3):
    print(f"  {r.doc_id}: {corpus[r.doc_id].title[:90]}")

Consulta PLAIN-1008: 'deafness'
Tokens de la consulta: ['deaf']

Algunos documentos relevantes en qrels:
  [1] MED-4532: Cytotoxic response of breast cancer cell lines, MCF 7 and T 47 D to triphala and its modif
  [1] MED-4533: Heavy metal content of ayurvedic herbal medicine products.
  [1] MED-4534: Triphala, Ayurvedic formulation for treating and preventing cancer: a review.

Top-3 de BM25:
  MED-4668: Cretinism revisited.


### Interpretación de los resultados

**Quién ganó**

- **BM25 con stemming** fue el mejor en las tres métricas: `P@10` = 0.241, `R@10` = 0.155 y `NDCG@10` = 0.330. TF-IDF con stemming sacó 0.231, 0.153 y 0.319.
- La ventaja es pequeña pero constante: BM25 gana en 97 consultas, TF-IDF en 52 y empatan en 174 (p < 0.01).
- El stemming mejora a los dos modelos (NDCG@10: +0.016 en TF-IDF y +0.018 en BM25).
- Los valores absolutos son bajos porque cada consulta tiene ~38 relevantes y solo miramos 10 resultados.

**Por qué gana BM25**

- **Saturación:** en BM25, repetir una palabra sube el score cada vez menos. Un documento no gana solo por repetir un término.
- **Longitud:** el coseno divide por todo el vector del documento, así que castiga a los documentos con muchos términos distintos. BM25 solo compara el largo con el promedio. El top-10 de TF-IDF es algo más corto (155 tokens) que el de BM25 (163), y los relevantes miden 167.
- Los documentos de NFCorpus tienen largo parecido, por eso la diferencia entre modelos es chica.

**Por qué a veces sacan 0**

- Pasa en 95 de 323 consultas, en los dos modelos a la vez.
- Los documentos relevantes usan sinónimos o términos médicos distintos a los de la consulta. Ejemplo: en `deafness`, los relevantes hablan de *triphala* y cáncer, y ninguno dice esa palabra.
- Los qrels de NFCorpus son indirectos (vienen de enlaces entre artículos de NutritionFacts).
- Ambos modelos solo comparan palabras exactas, no entienden sinónimos.

**Conclusión:** el servicio web usa BM25 con stemming como mejor opción, aunque también ofrece TF-IDF.

## 5. Prueba del servicio web

Uso `TestClient` de FastAPI (con `with`, para que corra el `lifespan` que indexa al iniciar). Se valida:

1. `GET /health`.
2. `POST /search` con `bm25` y `tfidf`: estructura, `rank` consecutivo, scores en orden descendente, sin duplicados, `top_k` respetado y `document_id` existente en el corpus.
3. Errores con `model` inválido y con `top_k <= 0`.
4. Que las métricas calculadas **con la API** den igual que en la sección 4. Eso confirma que los `document_id` coinciden con los qrels.

In [13]:
from fastapi.testclient import TestClient
from app import app

CAMPOS = {"rank", "document_id", "score", "title", "snippet"}

def validar(body: dict, modelo: str, top_k: int) -> None:
    assert set(body) == {"model", "results"} and body["model"] == modelo
    res = body["results"]
    assert 0 < len(res) <= top_k
    assert [r["rank"] for r in res] == list(range(1, len(res) + 1))
    assert all(set(r) == CAMPOS for r in res)
    scores = [r["score"] for r in res]
    assert scores == sorted(scores, reverse=True)
    ids = [r["document_id"] for r in res]
    assert len(ids) == len(set(ids)) and all(i in corpus for i in ids)
    assert all(len(r["snippet"]) <= 250 for r in res)

with TestClient(app) as client:
    salud = client.get("/health")
    print("GET /health ->", salud.status_code, salud.json())
    assert salud.json() == {"status": "ok", "corpus": "nfcorpus",
                            "documents": len(corpus), "models": ["tfidf", "bm25"]}

    for modelo in ("bm25", "tfidf"):
        r = client.post("/search", json={"query": "statins and breast cancer", "model": modelo, "top_k": 5})
        validar(r.json(), modelo, 5)
        print(f"\nPOST /search ({modelo}) -> {r.status_code}, estructura OK. Primeros 2 resultados:")
        print(json.dumps({"model": r.json()["model"], "results": r.json()["results"][:2]},
                         indent=2, ensure_ascii=False))

    print("\nErrores esperados:")
    invalidos = [{"query": "cancer", "model": "dense", "top_k": 10},
                 {"query": "cancer", "model": "bm25", "top_k": 0},
                 {"query": "cancer", "model": "bm25", "top_k": -3}]
    for payload in invalidos:
        codigo = client.post("/search", json=payload).status_code
        assert codigo in (400, 422)
        print(f"  {payload} -> HTTP {codigo}")

    # Métricas calculadas a través de la API
    from src.evaluation import ndcg_at_k, precision_at_k, recall_at_k
    for modelo in ("bm25", "tfidf"):
        p, r_, n = [], [], []
        for qid in ids_evaluados:
            resp = client.post("/search", json={"query": queries[qid], "model": modelo, "top_k": 10})
            ranking = [x["document_id"] for x in resp.json()["results"]]
            p.append(precision_at_k(ranking, test_qrels[qid])); r_.append(recall_at_k(ranking, test_qrels[qid]))
            n.append(ndcg_at_k(ranking, test_qrels[qid]))
        offline = summarize(resultados[(modelo, True)])
        via_api = {"P@10": np.mean(p), "R@10": np.mean(r_), "NDCG@10": np.mean(n)}
        assert all(abs(offline[m] - via_api[m]) < 1e-3 for m in offline)
        print(f"\n{modelo.upper()} vía API sobre {len(ids_evaluados)} consultas:",
              {m: round(float(v), 4) for m, v in via_api.items()})
print("\nTodas las validaciones de la API pasaron.")

GET /health -> 200 {'status': 'ok', 'corpus': 'nfcorpus', 'documents': 3633, 'models': ['tfidf', 'bm25']}

POST /search (bm25) -> 200, estructura OK. Primeros 2 resultados:
{
  "model": "bm25",
  "results": [
    {
      "rank": 1,
      "document_id": "MED-10",
      "score": 19.112451,
      "title": "Statin Use and Breast Cancer Survival: A Nationwide Cohort Study from Finland",
      "snippet": "Recent studies have suggested that statins, an established drug group in the prevention of cardiovascular mortality, could delay or prevent breast cancer recurrence but the effect on disease-specific mortality remains unclear. We evaluated risk of br"
    },
    {
      "rank": 2,
      "document_id": "MED-14",
      "score": 18.634513,
      "title": "Statin use after diagnosis of breast cancer and survival: a population-based cohort study.",
      "snippet": "BACKGROUND: Preclinical studies have shown that statins, particularly simvastatin, can prevent growth in breast cancer cell lines a


BM25 vía API sobre 323 consultas: {'P@10': 0.2406, 'R@10': 0.1549, 'NDCG@10': 0.3297}



TFIDF vía API sobre 323 consultas: {'P@10': 0.231, 'R@10': 0.1526, 'NDCG@10': 0.3193}

Todas las validaciones de la API pasaron.


In [14]:
# Pruebas automáticas del proyecto (tests/test_api.py)
import subprocess
salida = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_api.py"],
                        cwd=ROOT, capture_output=True, text=True)
print(salida.stdout[-600:])
assert salida.returncode == 0

.............                                                            [100%]
13 passed in 4.35s



## 6. Cómo reproducir

```bash
python -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt pytest httpx
pytest -q                              # pruebas de la API
uvicorn app:app --port 10000           # servicio local en http://127.0.0.1:10000
```

El despliegue en Render (Docker) está explicado en el `README.md`.